In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import time

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import joblib

PROJECT_ROOT = Path.cwd()
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
OUTPUT_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else PROJECT_ROOT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR_CANDIDATES = [
    Path(os.environ['CIC_IDS_DATA_DIR']) if os.environ.get('CIC_IDS_DATA_DIR') else None,
    PROJECT_ROOT,
    PROJECT_ROOT / 'datasets',
    KAGGLE_INPUT,
    KAGGLE_INPUT / 'compressed-cic-2018',
    KAGGLE_WORKING,
]
DATA_DIR_CANDIDATES = [p for p in DATA_DIR_CANDIDATES if p is not None]


def find_data_file(filename: str) -> Path:
    for base in DATA_DIR_CANDIDATES:
        candidate = base / filename
        if candidate.exists():
            return candidate
    searched = '\n'.join(str(p / filename) for p in DATA_DIR_CANDIDATES)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


def artifact_path(filename: str) -> Path:
    return OUTPUT_DIR / filename


def hierarchical_f1(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    f1_binary = f1_score((y_true > 0), (y_pred > 0), average='macro', zero_division=0)

    attack_mask = y_true > 0
    f1_attack = f1_score(
        y_true[attack_mask],
        y_pred[attack_mask],
        average='macro',
        zero_division=0
    ) if attack_mask.any() else 0.0

    return min(f1_binary, f1_attack)


def add_binary_metrics(y_true, y_pred):
    y_true_bin = (np.asarray(y_true) != 0).astype(int)
    y_pred_bin = (np.asarray(y_pred) != 0).astype(int)
    return {
        'multiclass_accuracy': accuracy_score(y_true, y_pred),
        'hierarchical_f1': hierarchical_f1(y_true, y_pred),
        'binary_precision': precision_score(y_true_bin, y_pred_bin, zero_division=0),
        'binary_recall': recall_score(y_true_bin, y_pred_bin, zero_division=0),
        'binary_f1': f1_score(y_true_bin, y_pred_bin, zero_division=0),
    }

/mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load compressed VAE data
train_compressed_path = find_data_file('df_train.csv')
test_compressed_path = find_data_file('df_test.csv')
df_train = pd.read_csv(train_compressed_path)
df_test = pd.read_csv(test_compressed_path)

print(f'Train path: {train_compressed_path}')
print(f'Test path: {test_compressed_path}')
print(f'Train shape: {df_train.shape}')
print(f'Test shape: {df_test.shape}')
print(f"\nTrain label distribution:\n{df_train['label'].value_counts()}")
print(f"\nTest label distribution:\n{df_test['label'].value_counts()}")

Train path: /mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/df_train.csv
Test path: /mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/df_test.csv
Train shape: (2049029, 8)
Test shape: (1048575, 8)

Train label distribution:
label
0    1663709
4     187587
3     145241
1      41508
2      10984
Name: count, dtype: int64

Test label distribution:
label
8    686012
0    360833
9      1730
Name: count, dtype: int64


In [3]:
# Prepare features and labels
feature_cols = ['latent_0', 'latent_1', 'latent_2', 'latent_3', 'latent_4', 'recon_loss', 'kld_loss']

X_train_full = df_train[feature_cols].values
y_train_full = df_train['label'].values

X_test = df_test[feature_cols].values
y_test = df_test['label'].values

# Compute inverse class weights
sample_weights_full = compute_sample_weight(class_weight='balanced', y=y_train_full)

print(f"X_train_full: {X_train_full.shape}, X_test: {X_test.shape}")
print(f"\nClass distribution in training:")
unique, counts = np.unique(y_train_full, return_counts=True)
for c, cnt in zip(unique, counts):
    print(f"  Class {c}: {cnt} samples, weight: {len(y_train_full) / (len(unique) * cnt):.4f}")

X_train_full: (2049029, 7), X_test: (1048575, 7)

Class distribution in training:
  Class 0: 1663709 samples, weight: 0.2463
  Class 1: 41508 samples, weight: 9.8729
  Class 2: 10984 samples, weight: 37.3093
  Class 3: 145241 samples, weight: 2.8216
  Class 4: 187587 samples, weight: 2.1846


# Optuna Hyperparameter Tuning

In [4]:
def objective(trial):
    params = {
        'objective': 'multiclass',
        'num_class': len(np.unique(y_train_full)),
        'metric': 'multi_logloss',
        'verbosity': -1,
        'device': 'cuda',
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 100.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 100.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 2.0),
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1_scores = []
    
    for train_idx, val_idx in skf.split(X_train_full, y_train_full):
        X_train_fold = X_train_full[train_idx]
        y_train_fold = y_train_full[train_idx]
        X_val_fold = X_train_full[val_idx]
        y_val_fold = y_train_full[val_idx]
        sw_train_fold = sample_weights_full[train_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train_fold, y_train_fold,
            sample_weight=sw_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=0)]
        )
        y_pred = model.predict(X_val_fold)
        f1_scores.append(hierarchical_f1(y_val_fold, y_pred))
    
    return np.mean(f1_scores)

# Run Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, n_jobs=1, show_progress_bar=True, catch=(Exception,))


print(f"\nBest trial hierarchical F1: {study.best_trial.value:.4f}")
print(f"Best params: {study.best_trial.params}")

[I 2026-05-17 12:34:32,648] A new study created in memory with name: no-name-05b4f7eb-c640-4edc-90e7-24ccf9ff9b04
  0%|          | 0/20 [2:08:01<?, ?it/s]


[W 2026-05-17 14:42:34,092] Trial 0 failed with parameters: {'n_estimators': 1386, 'max_depth': 13, 'learning_rate': 0.0029096587215944624, 'num_leaves': 186, 'subsample': 0.6783011396290337, 'colsample_bytree': 0.8465670963664158, 'min_child_samples': 12, 'min_child_weight': 0.018257472278935233, 'reg_alpha': 1.7205932039223874e-06, 'reg_lambda': 0.008439323970707907, 'min_split_gain': 1.9706207272695804} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_92065/2130024391.py", line 33, in objective
    model.fit(
  File "/mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py", line 1560, in fit
    super().fit(
  File "/mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/.venv/lib/p

KeyboardInterrupt: 

# Train Final Model with Best Params

In [ ]:
best_params = study.best_trial.params
best_params.update({
    'objective': 'multiclass',
    'num_class': len(np.unique(y_train_full)),
    'metric': 'multi_logloss',
    'verbosity': -1,
    'device': 'cuda',
    'random_state': 42,
})

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_train_full, y_train_full, sample_weight=sample_weights_full)

print("Final model trained on full training data with class balancing!")


KeyboardInterrupt



# Final Evaluation on Test Set

In [ ]:
# Evaluate on test set
y_pred_test = final_model.predict(X_test)

# Compute sample weights for test set (multiclass)
sample_weights_test = compute_sample_weight(class_weight='balanced', y=y_test)

# Create binary labels for test: 0 = benign, 1 = attack (any non-0)
y_test_binary = (y_test != 0).astype(int)
y_pred_binary = (y_pred_test != 0).astype(int)

# Compute binary class weights (inverse frequency for 0 vs 1)
sample_weights_binary = compute_sample_weight(class_weight='balanced', y=y_test_binary)

# Custom weighted binary accuracy
def weighted_binary_accuracy(y_true_bin, y_pred_bin, sample_weights):
    correct_mask = y_true_bin == y_pred_bin
    
    benign_mask = y_true_bin == 0
    attack_mask = y_true_bin == 1
    
    benign_correct_weighted = np.sum(sample_weights[benign_mask & correct_mask])
    benign_total_weighted = np.sum(sample_weights[benign_mask])
    
    attack_correct_weighted = np.sum(sample_weights[attack_mask & correct_mask])
    attack_total_weighted = np.sum(sample_weights[attack_mask])
    
    total_correct_weighted = benign_correct_weighted + attack_correct_weighted
    total_weighted = np.sum(sample_weights)
    
    print(f"\nWeighted Binary Evaluation (Attack vs Benign):")
    print(f"  Benign (0) weighted acc: {benign_correct_weighted:.2f}/{benign_total_weighted:.2f} = {benign_correct_weighted/benign_total_weighted:.4f}")
    print(f"  Attack (1) weighted acc: {attack_correct_weighted:.2f}/{attack_total_weighted:.2f} = {attack_correct_weighted/attack_total_weighted:.4f}")
    
    return total_correct_weighted / total_weighted

print("=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)

# Binary class distribution
n_benign = np.sum(y_test_binary == 0)
n_attack = np.sum(y_test_binary == 1)
print(f"\nBinary test distribution: Benign={n_benign}, Attack={n_attack}")
print(f"Binary weights: Benign={len(y_test_binary)/(2*n_benign):.4f}, Attack={len(y_test_binary)/(2*n_attack):.4f}")

print(f"\nAccuracy (multiclass):  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Accuracy (multiclass weighted): {accuracy_score(y_test, y_pred_test, sample_weight=sample_weights_test):.4f}")
print(f"Accuracy (binary): {accuracy_score(y_test_binary, y_pred_binary):.4f}")
weighted_bin_acc = weighted_binary_accuracy(y_test_binary, y_pred_binary, sample_weights_binary)
print(f"Accuracy (binary weighted): {weighted_bin_acc:.4f}")

print(f"\nHierarchical F1: {hierarchical_f1(y_test, y_pred_test):.4f}")
print(f"Precision (macro): {precision_score(y_test, y_pred_test, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_test, y_pred_test, average='macro'):.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Binary: Attack vs Benign)")
print("=" * 60)
print(classification_report(y_test_binary, y_pred_binary, target_names=['Benign', 'Attack']))

In [ ]:
# Save the model
model_path = artifact_path('lightgbm_model.txt')
final_model.booster_.save_model(str(model_path))
print(f'Model saved to {model_path}')

In [ ]:
# Benchmark helpers: tuning functions and shared state
import gc
import time

latent_cols = [c for c in feature_cols if c.startswith('latent_')]
benchmark_rows = []
benchmark_models = {}
benchmark_predictions = {}


def build_lgb_params(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }


def tune_feature_set_lgb(X_train, y_train):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def objective(trial):
        params = build_lgb_params(trial)
        params.update({
            'objective': 'multiclass',
            'num_class': len(np.unique(y_train)),
            'metric': 'multi_logloss',
            'verbosity': -1,
            'device': 'cuda',
            'random_state': 42,
        })
        fold_scores = []

        for train_idx, val_idx in skf.split(X_train, y_train):
            X_fold_train = X_train[train_idx]
            y_fold_train = y_train[train_idx]
            X_fold_val = X_train[val_idx]
            y_fold_val = y_train[val_idx]
            fold_weights = compute_sample_weight(class_weight='balanced', y=y_fold_train)

            model = lgb.LGBMClassifier(**params)
            model.fit(
                X_fold_train, y_fold_train,
                sample_weight=fold_weights,
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=0)]
            )
            y_fold_pred = model.predict(X_fold_val)
            fold_scores.append(hierarchical_f1(y_fold_val, y_fold_pred))

        return float(np.mean(fold_scores))

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=20, show_progress_bar=True, catch=(Exception,))
    return study


def run_feature_set_benchmark_lgb(name, X_train, y_train, X_test, y_test):
    print("=" * 60)
    print(f"BENCHMARK: {name}")
    print("=" * 60)
    print(f"Training shape: {X_train.shape}")
    print(f"Test shape: {X_test.shape}")

    gc.collect()
    tuning_start = time.time()
    study = tune_feature_set_lgb(X_train, y_train)
    tuning_seconds = time.time() - tuning_start

    best_params = study.best_trial.params.copy()
    best_params.update({
        'objective': 'multiclass',
        'num_class': len(np.unique(y_train)),
        'metric': 'multi_logloss',
        'verbosity': -1,
        'device': 'cuda',
        'random_state': 42,
    })

    model = lgb.LGBMClassifier(**best_params)
    fit_start = time.time()
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    model.fit(X_train, y_train, sample_weight=sample_weights)
    fit_seconds = time.time() - fit_start

    y_pred = model.predict(X_test)
    y_test_binary = (np.asarray(y_test) != 0).astype(int)
    y_pred_binary = (np.asarray(y_pred) != 0).astype(int)
    sample_weights_binary = compute_sample_weight(class_weight='balanced', y=y_test_binary)
    metrics = add_binary_metrics(y_test, y_pred)

    print("\n" + "=" * 60)
    print("FINAL TEST SET EVALUATION")
    print("=" * 60)
    print(f"\nBinary test distribution: Benign={np.sum(y_test_binary == 0)}, Attack={np.sum(y_test_binary == 1)}")
    print(f"Binary weights: Benign={len(y_test_binary)/(2*np.sum(y_test_binary == 0)):.4f}, Attack={len(y_test_binary)/(2*np.sum(y_test_binary == 1)):.4f}")
    print(f"\nAccuracy (multiclass):  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Accuracy (multiclass weighted): {accuracy_score(y_test, y_pred, sample_weight=compute_sample_weight(class_weight='balanced', y=y_test)):.4f}")
    print(f"Accuracy (binary): {accuracy_score(y_test_binary, y_pred_binary):.4f}")
    print("\nWeighted Binary Evaluation (Attack vs Benign):")
    correct_mask = y_test_binary == y_pred_binary
    benign_mask = y_test_binary == 0
    attack_mask = y_test_binary == 1
    benign_correct_weighted = np.sum(sample_weights_binary[benign_mask & correct_mask])
    benign_total_weighted = np.sum(sample_weights_binary[benign_mask])
    attack_correct_weighted = np.sum(sample_weights_binary[attack_mask & correct_mask])
    attack_total_weighted = np.sum(sample_weights_binary[attack_mask])
    total_correct_weighted = benign_correct_weighted + attack_correct_weighted
    total_weighted = np.sum(sample_weights_binary)
    print(f"  Benign (0) weighted acc: {benign_correct_weighted:.2f}/{benign_total_weighted:.2f} = {benign_correct_weighted/benign_total_weighted:.4f}")
    print(f"  Attack (1) weighted acc: {attack_correct_weighted:.2f}/{attack_total_weighted:.2f} = {attack_correct_weighted/attack_total_weighted:.4f}")
    print(f"Accuracy (binary weighted): {total_correct_weighted / total_weighted:.4f}")
    print(f"\nHierarchical F1: {hierarchical_f1(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")

    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT (Binary: Attack vs Benign)")
    print("=" * 60)
    print(classification_report(y_test_binary, y_pred_binary, target_names=['Benign', 'Attack']))

    row = {
        'feature_set': name,
        'tuning_seconds': tuning_seconds,
        'fit_seconds': fit_seconds,
        'train_seconds': tuning_seconds + fit_seconds,
        'best_trial_hierarchical_f1': study.best_trial.value,
        'n_trials': 20,
    }
    row.update(metrics)

    benchmark_rows.append(row)
    benchmark_models[name] = model
    benchmark_predictions[name] = y_pred

    return row


def add_pretrained_model_benchmark_lgb(name, model_obj, X_test, y_test):
    """Add results from an already-trained model (like final_model) to benchmark."""
    print("=" * 60)
    print(f"BENCHMARK: {name} (using pre-trained final model)")
    print("=" * 60)
    print(f"Test shape: {X_test.shape}")

    y_pred = model_obj.predict(X_test)
    y_test_binary = (np.asarray(y_test) != 0).astype(int)
    y_pred_binary = (np.asarray(y_pred) != 0).astype(int)
    sample_weights_binary = compute_sample_weight(class_weight='balanced', y=y_test_binary)
    metrics = add_binary_metrics(y_test, y_pred)

    print("\n" + "=" * 60)
    print("FINAL TEST SET EVALUATION")
    print("=" * 60)
    print(f"\nBinary test distribution: Benign={np.sum(y_test_binary == 0)}, Attack={np.sum(y_test_binary == 1)}")
    print(f"Binary weights: Benign={len(y_test_binary)/(2*np.sum(y_test_binary == 0)):.4f}, Attack={len(y_test_binary)/(2*np.sum(y_test_binary == 1)):.4f}")
    print(f"\nAccuracy (multiclass):  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Accuracy (multiclass weighted): {accuracy_score(y_test, y_pred, sample_weight=compute_sample_weight(class_weight='balanced', y=y_test)):.4f}")
    print(f"Accuracy (binary): {accuracy_score(y_test_binary, y_pred_binary):.4f}")
    print("\nWeighted Binary Evaluation (Attack vs Benign):")
    correct_mask = y_test_binary == y_pred_binary
    benign_mask = y_test_binary == 0
    attack_mask = y_test_binary == 1
    benign_correct_weighted = np.sum(sample_weights_binary[benign_mask & correct_mask])
    benign_total_weighted = np.sum(sample_weights_binary[benign_mask])
    attack_correct_weighted = np.sum(sample_weights_binary[attack_mask & correct_mask])
    attack_total_weighted = np.sum(sample_weights_binary[attack_mask])
    total_correct_weighted = benign_correct_weighted + attack_correct_weighted
    total_weighted = np.sum(sample_weights_binary)
    print(f"  Benign (0) weighted acc: {benign_correct_weighted:.2f}/{benign_total_weighted:.2f} = {benign_correct_weighted/benign_total_weighted:.4f}")
    print(f"  Attack (1) weighted acc: {attack_correct_weighted:.2f}/{attack_total_weighted:.2f} = {attack_correct_weighted/attack_total_weighted:.4f}")
    print(f"Accuracy (binary weighted): {total_correct_weighted / total_weighted:.4f}")
    print(f"\nHierarchical F1: {hierarchical_f1(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")

    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT (Binary: Attack vs Benign)")
    print("=" * 60)
    print(classification_report(y_test_binary, y_pred_binary, target_names=['Benign', 'Attack']))

    row = {
        'feature_set': name,
        'tuning_seconds': 0,
        'fit_seconds': 0,
        'train_seconds': 0,
        'best_trial_hierarchical_f1': 0,
        'n_trials': 0,
    }
    row.update(metrics)

    benchmark_rows.append(row)
    benchmark_models[name] = model_obj
    benchmark_predictions[name] = y_pred

    return row

In [ ]:
compressed_loss_result = add_pretrained_model_benchmark_lgb(
    'compressed + recon/kld loss',
    final_model,
    df_test[feature_cols].values,
    y_test,
)


In [ ]:
compressed_latent_result = run_feature_set_benchmark_lgb(
    'compressed latent only',
    df_train[latent_cols].values,
    y_train_full,
    df_test[latent_cols].values,
    y_test,
)


In [ ]:
full_original_result = run_feature_set_benchmark_lgb(
    'full original features',
    df_train.drop(columns=['label']).values,
    df_train['label'].values,
    df_test.drop(columns=['label']).values,
    df_test['label'].values,
)


In [ ]:
benchmark_df = pd.DataFrame(benchmark_rows).sort_values('binary_f1', ascending=False)
print(benchmark_df)
benchmark_df.to_csv(artifact_path('lightgbm_feature_set_benchmark.csv'), index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=benchmark_df, x='feature_set', y='train_seconds', ax=axes[0])
axes[0].set_title('Training Time by Feature Set')
axes[0].set_xlabel('Feature set')
axes[0].set_ylabel('Seconds')
axes[0].tick_params(axis='x', rotation=20)

plot_df = benchmark_df.melt(
    id_vars=['feature_set'],
    value_vars=['hierarchical_f1', 'binary_f1', 'binary_precision', 'binary_recall'],
    var_name='metric',
    value_name='score',
)
sns.barplot(data=plot_df, x='feature_set', y='score', hue='metric', ax=axes[1])
axes[1].set_title('Benchmark Metrics by Feature Set')
axes[1].set_xlabel('Feature set')
axes[1].set_ylabel('Score')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(title='metric')

plt.tight_layout()
plt.show()